In [1]:
# Install 
import subprocess
subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn", "pyngrok",
                "python-multipart", "nest-asyncio", "transformers",
                "accelerate", "bitsandbytes"], check=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00


CompletedProcess(args=['pip', 'install', '-q', 'fastapi', 'uvicorn', 'pyngrok', 'python-multipart', 'nest-asyncio', 'transformers', 'accelerate', 'bitsandbytes'], returncode=0)

In [2]:
# Imports
import io, base64, time, threading
import torch, uvicorn, nest_asyncio
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from pyngrok import ngrok, conf
from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    BitsAndBytesConfig,
)

In [3]:
# Config 
NGROK_TOKEN = ""
MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"  # LLaVA-NeXT
USE_4BIT    = True   # False = fp16 ~14GB | True = 4-bit ~4GB
PORT        = 8000
DEVICE      = "cuda:0"

# Load model 
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Loading {MODEL_ID} ({'4-bit' if USE_4BIT else 'fp16'})...")

quant_cfg = None
dtype     = torch.float16
if USE_4BIT:
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    dtype = None
    
    processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    quantization_config=quant_cfg,
    device_map={"": 0},
    low_cpu_mem_usage=True,
).eval()

used  = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Model loaded ✓  VRAM: {used:.1f} / {total:.1f} GB")


GPU : Tesla T4
VRAM: 15.6 GB
Loading llava-hf/llava-v1.6-mistral-7b-hf (4-bit)...


processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded ✓  VRAM: 4.3 / 15.6 GB


In [4]:
# Prompts 
# Prompt 1: Identity check (2 images — reference + live frame)
IDENTITY_PROMPT = (
    "USER: <image>\n<image>\n"
    "Image 1 is the registered ID photo of the interview candidate. "
    "Image 2 is the current live camera frame from the interview session. "
    "Are both images showing the SAME person? "
    "Reply with only YES or NO.\n"
    "ASSISTANT:"
)

# Prompt 2: Cheat detection (1 image — live frame only)
CHEAT_PROMPT = (
    "USER: <image>\n"
    "This is a live camera frame from an online interview session. "
    "Carefully analyze the image and check for ANY of the following cheating behaviors:\n"
    "1. More than one person visible in the frame\n"
    "2. Person looking away from screen/camera repeatedly or suspiciously\n"
    "3. Person reading from physical notes, papers, or books\n"
    "4. A phone, tablet, or second screen visible\n"
    "5. Someone whispering or prompting from off-screen\n"
    "6. Person wearing earphones/earbuds that could be used for prompting\n"
    "7. No person visible (left the frame)\n"
    "If ANY cheating behavior is detected, reply in this exact format:\n"
    "CHEAT: <short description of what you detected>\n"
    "If everything looks normal and the candidate appears to be attending honestly, reply with:\n"
    "CLEAN\n"
    "ASSISTANT:"
)

In [5]:
# Inference helpers 
lock = threading.Lock()

@torch.inference_mode()
def _generate(prompt: str, images: list) -> str:
    inputs = processor(text=prompt, images=images, return_tensors="pt").to(DEVICE)
    if not USE_4BIT:
        inputs = {k: v.half() if v.dtype == torch.float32 else v for k, v in inputs.items()}
    out_ids = model.generate(
        **inputs, max_new_tokens=60, do_sample=False, temperature=None, top_p=None
    )
    new_tok = out_ids[0][inputs["input_ids"].shape[1]:]
    return processor.decode(new_tok, skip_special_tokens=True).strip()


def check_identity(ref_pil: Image.Image, frame_pil: Image.Image) -> dict:
    t0     = time.perf_counter()
    answer = _generate(IDENTITY_PROMPT, [ref_pil, frame_pil]).upper()
    return {
        "match":     answer.startswith("YES"),
        "raw":       answer,
        "elapsed_s": round(time.perf_counter() - t0, 3),
    }


def check_cheating(frame_pil: Image.Image) -> dict:
    t0     = time.perf_counter()
    answer = _generate(CHEAT_PROMPT, [frame_pil])
    upper  = answer.upper()

    if upper.startswith("CHEAT"):
        # Extract description after "CHEAT: "
        description = answer[answer.find(":")+1:].strip() if ":" in answer else answer
        cheating    = True
    else:
        description = "No cheating detected"
        cheating    = False

    return {
        "cheating":    cheating,
        "description": description,
        "raw":         answer,
        "elapsed_s":   round(time.perf_counter() - t0, 3),
    }

In [6]:
# FastAPI
app = FastAPI(title="Interview Proctoring API")

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model":  MODEL_ID,
        "vram_used_gb": round(torch.cuda.memory_allocated(0) / 1e9, 2),
        "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    }


@app.post("/analyze")
async def analyze(
    reference: UploadFile = File(..., description="Candidate's registered ID photo"),
    frame:     UploadFile = File(..., description="Current live webcam frame"),
):
    """
    Main endpoint — runs BOTH checks in one call.

    POST multipart/form-data:
        reference : candidate ID image
        frame     : live webcam frame

    Returns:
    {
        "identity": {
            "match":     bool,      # True = same person as registered
            "raw":       str,
            "elapsed_s": float
        },
        "cheating": {
            "cheating":    bool,    # True = cheat detected
            "description": str,     # what was detected
            "raw":         str,
            "elapsed_s":   float
        },
        "alert":        bool,       # True if EITHER check failed
        "alert_reason": str,        # human-readable summary
        "total_elapsed_s": float
    }
    """
    try:
        ref_pil   = Image.open(io.BytesIO(await reference.read())).convert("RGB")
        frame_pil = Image.open(io.BytesIO(await frame.read())).convert("RGB")
    except Exception as e:
        raise HTTPException(400, f"Image decode failed: {e}")

    t_total = time.perf_counter()
    with lock:
        identity = check_identity(ref_pil, frame_pil)
        cheating = check_cheating(frame_pil)
    total = round(time.perf_counter() - t_total, 3)

    # Build alert summary
    alert        = (not identity["match"]) or cheating["cheating"]
    alert_reason = []
    if not identity["match"]:
        alert_reason.append("Identity mismatch — different person detected")
    if cheating["cheating"]:
        alert_reason.append(f"Cheating detected — {cheating['description']}")
    alert_reason = " | ".join(alert_reason) if alert_reason else "All clear"

    result = {
        "identity":        identity,
        "cheating":        cheating,
        "alert":           alert,
        "alert_reason":    alert_reason,
        "total_elapsed_s": total,
    }

    flag = "ALERT" if alert else "OK"
    print(f"[{flag}] identity={identity['match']}  cheat={cheating['cheating']}  "
          f"reason='{alert_reason}'  total={total}s")

    return JSONResponse(result)


@app.post("/analyze_base64")
async def analyze_base64(payload: dict):
    """
    Same as /analyze but accepts JSON with base64 images.
    {
        "reference": "<base64>",
        "frame":     "<base64>"
    }
    """
    try:
        ref_pil   = Image.open(io.BytesIO(base64.b64decode(payload["reference"]))).convert("RGB")
        frame_pil = Image.open(io.BytesIO(base64.b64decode(payload["frame"]))).convert("RGB")
    except Exception as e:
        raise HTTPException(400, f"Decode failed: {e}")

    t_total = time.perf_counter()
    with lock:
        identity = check_identity(ref_pil, frame_pil)
        cheating = check_cheating(frame_pil)
    total = round(time.perf_counter() - t_total, 3)

    alert        = (not identity["match"]) or cheating["cheating"]
    alert_reason = []
    if not identity["match"]:
        alert_reason.append("Identity mismatch — different person detected")
    if cheating["cheating"]:
        alert_reason.append(f"Cheating detected — {cheating['description']}")
    alert_reason = " | ".join(alert_reason) if alert_reason else "All clear"

    return JSONResponse({
        "identity":        identity,
        "cheating":        cheating,
        "alert":           alert,
        "alert_reason":    alert_reason,
        "total_elapsed_s": total,
    })

In [8]:
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(PORT, bind_tls=True)
print(f"API URL -> {public_url.public_url}")

API URL -> https://unenunciative-jackson-nonsupportably.ngrok-free.dev


In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=PORT)
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())

INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     38.183.101.34:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     38.183.101.34:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     38.183.101.34:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     38.183.101.34:0 - "GET /openapi.json HTTP/1.1" 200 OK
